# RAG Pipeline- Data Ingestion to vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/adityakulkarni/Downloads/AI Engineer🤖/RAG/RAG_Basic/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Ingestion

In [3]:
### Read all pdfs in dir
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF flies to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # adding extra metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"All docs loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 3 PDF flies to process

Processing: vector.pdf
Loaded 65 pages

Processing: rag.pdf
Loaded 25 pages

Processing: rag_basics.pdf
Loaded 1 pages
All docs loaded: 91


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 25.1.182', 'creator': 'Acrobat PDFMaker 25 for PowerPoint', 'creationdate': '2025-11-19T09:22:24-05:00', 'author': 'ken', 'moddate': '2025-11-19T09:22:31-05:00', 'title': 'CS 5412:  Topics in Cloud Computing', 'source': '../data/pdf_files/vector.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1', 'source_file': 'vector.pdf', 'file_type': 'pdf'}, page_content='VECTOR DATABASES\nCS4414/CS5416 \nLecture 24\nHTTP://WWW.CS.CORNELL.EDU/COURSES/CS5412/2025FA 1'),
 Document(metadata={'producer': 'Adobe PDF Library 25.1.182', 'creator': 'Acrobat PDFMaker 25 for PowerPoint', 'creationdate': '2025-11-19T09:22:24-05:00', 'author': 'ken', 'moddate': '2025-11-19T09:22:31-05:00', 'title': 'CS 5412:  Topics in Cloud Computing', 'source': '../data/pdf_files/vector.pdf', 'total_pages': 65, 'page': 1, 'page_label': '2', 'source_file': 'vector.pdf', 'file_type': 'pdf'}, page_content='IDEA MAP FOR TODAY\nHTTP://WWW.CS.CORNELL.EDU/COURSES/CS5412/2025FA 2

In [5]:
### Text splitting get into chunks

def split_dcuments(documents,chunk_size= 1000,chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

chunks = split_dcuments(all_pdf_documents)


Split 91 documents into 158 chunks


### Embeddings & VectorStore DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        #Load Sentence transformer model
        try:
            print(f"Loading embeddings model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embeddings dimension: {self.model}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not Loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddngs with shape: {embeddings.shape}")
        return embeddings
    

## init embeddings manager

embeddings_manager = EmbeddingManager()

Loading embeddings model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6724.57it/s]


Model Loaded successfully. Embeddings dimension: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


### Vector Store

In [8]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF Document embeddings for RAG"}
            )

            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing Documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise
    
    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of Documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embeddings.tolist())

        try:
            self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
            print(f"Successfullt added {len(documents)} documents to vector store")
            print (f"Total Documents in collection: {self.collection.count()}")

        except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise

vectorstore = VectorStore()

Vector Store initialized. Collection: pdf_documents
Existing Documents in collection: 316


In [9]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]


## GEerate EMbeddigns
embeddigns = embeddings_manager.generate_embeddings(texts)

##Store in VectorDB
vectorstore.add_documents(chunks, embeddigns)

Generating embeddings for 158 texts...


Batches: 100%|██████████| 5/5 [00:01<00:00,  3.84it/s]


Generated Embeddngs with shape: (158, 384)
Adding 158 documents to vector store...
Successfullt added 158 documents to vector store
Total Documents in collection: 474


# Retriver Pipeline from VectorStore

In [10]:
class RAGRetriver:

    def __init__(self, vector_store: VectorStore, embeddings_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embeddings_manager = embeddings_manager

    def retriver(self, query:str, top_k:int = 5, score_threshold: float= 0.0) -> List[Dict[str, Any]]:

        print(f"Retriving Documetns for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embeddings_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrived {len(retrived_docs)} documents after filtering")
            else:
                print("No docs found") 
            
            return retrived_docs           

        except Exception as e:
            print(f"Error converting query to Embeddings: {e}")
            return []
        
rag_retriver = RAGRetriver(vectorstore, embeddings_manager)


In [11]:
rag_retriver.retriver("what is Information Retrieval")

Retriving Documetns for query: 'what is Information Retrieval'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.21it/s]

Generated Embeddngs with shape: (1, 384)
retrived 5 documents after filtering


[{'id': 'doc_d2ae314d_75',
  'content': 'retrieval\nIR all manner of media based on user information needs. The resulting IR system is\noften called a search engine. Our goal in this section is to give a sufﬁcient overview\nof IR to see its application to large language models meeting user information needs.\nReaders with more interest speciﬁcally in information retrieval should see the His-\ntorical Notes section at the end of the chapter.\nThe IR task we consider is called ad hoc retrieval , in which a user poses aad hoc retrieval\nquery to a retrieval system, which then returns an ordered set of documents from\nsome collection. A document refers to whatever unit of text the system indexesdocument\nand retrieves (web pages, scientiﬁc papers, news articles, or even shorter passages\nlike paragraphs). A collection refers to a set of documents being used to satisfycollection\nuser requests. A collection can mean the entire web, in which case we are doing',
  'metadata': {'subject': '',


# Integration VectorDB COntent Pipeline with LLM Output

In [12]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# Simple RAG function
def rag_simple(query, retriver, top_k=3):
    results = retriver.retriver(query, top_k=top_k)

    # build context
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found"

    prompt = f"""
    You are a helpful AI assistant.

    Answer the question using ONLY the context below.
    If the answer is not present in the context, say "I don't know".

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=1024
    )


    return response.choices[0].message.content

answer = rag_simple("What is a vector database?", rag_retriver)
print(answer)

Retriving Documetns for query: 'What is a vector database?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.05it/s]

Generated Embeddngs with shape: (1, 384)
retrived 3 documents after filtering


I don't know.
